# EDA — Physics Problems (Dataset 2026-05-15)

> **Dataset:** `EXACT_Materials/Datasets/EXACT2026_dataset_2026-05-15/Physics_Problems_Text_Only.csv`  
> **Rows:** 1,352 | **Columns:** `id`, `question`, `cot`, `answer`, `unit`  
> **Date:** 2026-05-17

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 120

%matplotlib inline

In [ ]:
# ── Load dataset ──
DATA_PATH = Path("../../../EXACT_Materials/Datasets/EXACT2026_dataset_2026-05-15/Physics_Problems_Text_Only/Physics_Problems_Text_Only.csv")
df = pd.read_csv(DATA_PATH)

# ── Derived columns ──
df["prefix"] = df["id"].str.extract(r"^([A-Z]+)")
df["answer_numeric"] = pd.to_numeric(df["answer"], errors="coerce")
df["q_len"] = df["question"].str.len()
df["q_words"] = df["question"].str.split().str.len()
df["cot_len"] = df["cot"].str.len()
df["cot_steps"] = df["cot"].str.count(r"Step \d+")

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

---
## 1. Basic Info & Null Check

In [ ]:
print("=== Data Types ===")
print(df[["id","question","cot","answer","unit"]].dtypes)
print()

print("=== Null Values ===")
null_counts = df[["id","question","cot","answer","unit"]].isnull().sum()
print(null_counts)
print()

print("=== Empty Strings ===")
for col in ["id","question","cot","answer","unit"]:
    empty = (df[col].astype(str).str.strip() == "").sum()
    if empty > 0:
        print(f"  {col}: {empty} empty strings")
    else:
        print(f"  {col}: OK")

In [ ]:
# 14 rows with null unit — need manual fix or BTC report
null_unit = df[df["unit"].isna()][["id", "question", "answer", "prefix"]].copy()
null_unit["question"] = null_unit["question"].str[:100] + "..."
print(f"Rows with null unit: {len(null_unit)}")
null_unit

---
## 2. Domain Distribution (ID Prefix)

In [ ]:
prefix_counts = df["prefix"].value_counts().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = sns.color_palette("viridis", len(prefix_counts))
prefix_counts.plot.barh(ax=axes[0], color=colors)
axes[0].set_xlabel("Number of Problems")
axes[0].set_ylabel("Domain (ID Prefix)")
axes[0].set_title("Problems per Domain")
for i, (val, name) in enumerate(zip(prefix_counts.values, prefix_counts.index)):
    axes[0].text(val + 5, i, f"{val} ({val/len(df)*100:.1f}%)", va="center", fontsize=9)

# Pie chart
prefix_counts_desc = prefix_counts.sort_values(ascending=False)
axes[1].pie(
    prefix_counts_desc.values, 
    labels=prefix_counts_desc.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=sns.color_palette("viridis", len(prefix_counts_desc))
)
axes[1].set_title("Domain Proportion")

plt.tight_layout()
plt.savefig("../../figures/physics/EDA_domain_distribution.png", bbox_inches="tight")
plt.show()

print("\nImbalance ratio (max/min):", prefix_counts.max() / prefix_counts.min())

---
## 3. Answer Type Classification

**Key finding:** 28% of answers are NOT pure numeric — scientific notation, text, Yes/No, multi-value, sqrt.

In [ ]:
def classify_answer(ans: str) -> str:
    """Classify answer into one of 7 categories."""
    ans = str(ans).strip()
    if ans in ("Yes", "No"):
        return "yes_no"
    try:
        float(ans)
        return "pure_numeric"
    except ValueError:
        pass
    if re.search(r"10\^|10[\u2070-\u2079\u207B]", ans):
        return "scientific_notation"
    if "sqrt" in ans or "\\sqrt" in ans:
        return "contains_sqrt"
    if ";" in ans:
        return "multi_value"
    if not any(c.isdigit() for c in ans):
        return "text_only"
    if any(c.isalpha() for c in ans):
        return "mixed_text_number"
    return "other"

df["answer_type"] = df["answer"].apply(classify_answer)

type_counts = df["answer_type"].value_counts()
print("=== Answer Type Distribution ===")
for t, c in type_counts.items():
    print(f"  {t:25s} {c:5d}  ({c/len(df)*100:5.1f}%)")
print(f"  {'TOTAL':25s} {len(df):5d}")
print(f"\n  Non-numeric total: {len(df) - type_counts.get('pure_numeric', 0)} ({(len(df) - type_counts.get('pure_numeric', 0))/len(df)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Overall answer type distribution
type_order = type_counts.index.tolist()
palette = {
    "pure_numeric": "#2ecc71", "scientific_notation": "#3498db",
    "text_only": "#e74c3c", "mixed_text_number": "#e67e22",
    "multi_value": "#9b59b6", "yes_no": "#1abc9c",
    "contains_sqrt": "#f39c12", "other": "#95a5a6"
}
colors_list = [palette.get(t, "#95a5a6") for t in type_order]
type_counts.plot.bar(ax=axes[0], color=colors_list)
axes[0].set_title("Answer Type Distribution")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=35)

# Stacked bar per domain
ct = pd.crosstab(df["prefix"], df["answer_type"])
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct.plot.barh(
    stacked=True, ax=axes[1],
    color=[palette.get(c, "#95a5a6") for c in ct_pct.columns]
)
axes[1].set_title("Answer Type % per Domain")
axes[1].set_xlabel("Percentage")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)

plt.tight_layout()
plt.savefig("../../figures/physics/EDA_answer_types.png", bbox_inches="tight")
plt.show()

In [ ]:
# Crosstab: answer type vs domain — key for pipeline routing
ct = pd.crosstab(df["prefix"], df["answer_type"], margins=True)
print("=== Answer Type × Domain Crosstab ===")
ct

### 3.1 Non-numeric Answer Samples

Show examples of each non-numeric type to understand what the model needs to output.

In [ ]:
for atype in ["scientific_notation", "text_only", "mixed_text_number", "multi_value", "yes_no", "contains_sqrt"]:
    sub = df[df["answer_type"] == atype][["id", "answer", "unit", "prefix"]].head(5)
    print(f"\n--- {atype} ({len(df[df['answer_type'] == atype])} total) ---")
    for _, row in sub.iterrows():
        print(f"  [{row['id']}] {row['answer']:50s}  unit={row['unit']}")

---
## 4. Unit Analysis

In [ ]:
unit_counts = df["unit"].value_counts(dropna=False)

print(f"Total unique units: {df['unit'].nunique()}")
print(f"Null units: {df['unit'].isna().sum()}")
print()

# Top 20 units
fig, ax = plt.subplots(figsize=(12, 6))
top_units = unit_counts.head(20)
top_units.plot.barh(ax=ax, color=sns.color_palette("viridis", len(top_units)))
ax.set_xlabel("Count")
ax.set_ylabel("Unit")
ax.set_title(f"Top 20 Units (of {df['unit'].nunique()} unique)")
ax.invert_yaxis()
for i, val in enumerate(top_units.values):
    ax.text(val + 2, i, str(val), va="center", fontsize=9)
plt.tight_layout()
plt.savefig("../../figures/physics/EDA_unit_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Unit inconsistencies ──

# 1. Dash variants: - (U+002D) vs — (U+2014)
dash_df = df[df["unit"].isin(["-", "\u2014"])]
print("=== Dash Unit Variants ===")
print(pd.crosstab(dash_df["prefix"], dash_df["unit"]))
print(f"Total dash-unit rows: {len(dash_df)}")
print()

# 2. Mu/micro: μ (U+03BC) vs µ (U+00B5)
mu_greek = df["unit"].str.contains(chr(956), na=False).sum()   # μ U+03BC
mu_micro = df["unit"].str.contains(chr(181), na=False).sum()   # µ U+00B5
print(f"=== Mu/Micro Inconsistency ===")
print(f"  \u03bcF (Greek mu U+03BC): {mu_greek}")
print(f"  \u00b5F (Micro sign U+00B5): {mu_micro}")
print()

# 3. Vietnamese remnants
viet_unit = df[df["unit"].str.contains("lần|Độ", na=False, regex=True)]
print(f"=== Vietnamese in Units: {len(viet_unit)} ===")
if len(viet_unit) > 0:
    print(viet_unit[["id", "answer", "unit"]].to_string())
print()

# 4. Multi-unit format
multi_unit = df[df["unit"].str.contains(";", na=False)]
print(f"=== Multi-Unit Answers: {len(multi_unit)} ===")
print(multi_unit["unit"].value_counts().to_string())

In [ ]:
# Heatmap: unit frequency per domain
# Only top 15 units to keep readable
top_15_units = df["unit"].value_counts().head(15).index.tolist()
sub = df[df["unit"].isin(top_15_units)]
ct_unit = pd.crosstab(sub["prefix"], sub["unit"])

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(ct_unit, annot=True, fmt="d", cmap="YlOrRd", ax=ax, linewidths=0.5)
ax.set_title("Unit Frequency per Domain (Top 15 Units)")
plt.tight_layout()
plt.savefig("../../figures/physics/EDA_unit_heatmap.png", bbox_inches="tight")
plt.show()

---
## 5. Numeric Answer Analysis

In [ ]:
numeric = df[df["answer_type"] == "pure_numeric"].copy()
print(f"Pure numeric answers: {len(numeric)} / {len(df)} ({len(numeric)/len(df)*100:.1f}%)")
print()
print("=== Descriptive Stats ===")
print(numeric["answer_numeric"].describe())
print()

# Magnitude distribution
print("=== Magnitude Distribution ===")
bins = [-np.inf, -1, 0, 1, 10, 100, 1000, 1e6, np.inf]
labels = ["<-1", "-1 to 0", "0 to 1", "1-10", "10-100", "100-1K", "1K-1M", ">1M"]
numeric["magnitude"] = pd.cut(numeric["answer_numeric"], bins=bins, labels=labels)
mag_counts = numeric["magnitude"].value_counts().sort_index()
for label, count in mag_counts.items():
    print(f"  {label:12s}: {count:4d}  ({count/len(numeric)*100:5.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Log-scale histogram of all numeric answers
valid = numeric["answer_numeric"][numeric["answer_numeric"] > 0]
axes[0].hist(np.log10(valid), bins=50, color="#3498db", edgecolor="white")
axes[0].set_xlabel("log10(answer)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of log10(answer) — Positive Only")
axes[0].axvline(x=0, color="red", linestyle="--", alpha=0.7, label="answer=1")
axes[0].legend()

# Box plot per domain
domain_order = numeric.groupby("prefix")["answer_numeric"].median().sort_values().index
sns.boxplot(
    data=numeric, x="prefix", y="answer_numeric",
    order=domain_order, ax=axes[1], showfliers=False
)
axes[1].set_title("Answer Range per Domain (no outliers)")
axes[1].set_ylabel("Answer Value")

# Magnitude bar chart
mag_counts.plot.bar(ax=axes[2], color=sns.color_palette("viridis", len(mag_counts)))
axes[2].set_title("Answer Magnitude Distribution")
axes[2].set_ylabel("Count")
axes[2].tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.savefig("../../figures/physics/EDA_numeric_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# Per-domain numeric stats
print("=== Numeric Answer Stats per Domain ===")
stats = numeric.groupby("prefix")["answer_numeric"].agg(["count","min","median","mean","max","std"])
stats = stats.round(2)
stats

---
## 6. Question & CoT Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Question word count distribution
axes[0, 0].hist(df["q_words"], bins=40, color="#2ecc71", edgecolor="white")
axes[0, 0].set_xlabel("Word Count")
axes[0, 0].set_ylabel("Frequency")
axes[0, 0].set_title("Question Length (words)")
axes[0, 0].axvline(df["q_words"].median(), color="red", linestyle="--", label=f"median={df['q_words'].median():.0f}")
axes[0, 0].legend()

# CoT length distribution
axes[0, 1].hist(df["cot_len"], bins=40, color="#3498db", edgecolor="white")
axes[0, 1].set_xlabel("Character Count")
axes[0, 1].set_ylabel("Frequency")
axes[0, 1].set_title("CoT Length (characters)")
axes[0, 1].axvline(df["cot_len"].median(), color="red", linestyle="--", label=f"median={df['cot_len'].median():.0f}")
axes[0, 1].legend()

# CoT steps distribution
axes[1, 0].hist(df["cot_steps"], bins=range(1, 15), color="#e74c3c", edgecolor="white", align="left")
axes[1, 0].set_xlabel("Number of Steps")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].set_title("CoT Step Count")
axes[1, 0].set_xticks(range(1, 14))

# Q words by domain
domain_order = df.groupby("prefix")["q_words"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="prefix", y="q_words", order=domain_order, ax=axes[1, 1])
axes[1, 1].set_title("Question Length per Domain")
axes[1, 1].set_ylabel("Word Count")

plt.tight_layout()
plt.savefig("../../figures/physics/EDA_question_cot_length.png", bbox_inches="tight")
plt.show()

print(f"Question — words: mean={df['q_words'].mean():.0f}, median={df['q_words'].median():.0f}, max={df['q_words'].max()}")
print(f"CoT — chars: mean={df['cot_len'].mean():.0f}, median={df['cot_len'].median():.0f}, max={df['cot_len'].max()}")
print(f"CoT — steps: mean={df['cot_steps'].mean():.1f}, median={df['cot_steps'].median():.0f}, max={df['cot_steps'].max()}")

In [ ]:
# CoT steps per domain — difficulty proxy
fig, ax = plt.subplots(figsize=(12, 5))

domain_order = df.groupby("prefix")["cot_steps"].mean().sort_values(ascending=False).index
sns.boxplot(data=df, x="prefix", y="cot_steps", order=domain_order, ax=ax, palette="coolwarm_r")
ax.set_title("CoT Steps per Domain (Difficulty Proxy)")
ax.set_ylabel("Number of Steps")
ax.set_xlabel("Domain")

# Annotate means
means = df.groupby("prefix")["cot_steps"].mean()
for i, prefix in enumerate(domain_order):
    ax.text(i, means[prefix] + 0.3, f"{means[prefix]:.1f}", ha="center", fontsize=9, color="red")

plt.tight_layout()
plt.savefig("../../figures/physics/EDA_difficulty_proxy.png", bbox_inches="tight")
plt.show()

print("=== Average CoT Steps (Difficulty Ranking) ===")
for prefix in domain_order:
    n = len(df[df["prefix"] == prefix])
    print(f"  {prefix:6s}: {means[prefix]:.1f} steps (n={n})")

---
## 7. Scientific Notation Deep-dive

237 answers use scientific notation — but in **multiple incompatible formats**.

In [ ]:
sci = df[df["answer_type"] == "scientific_notation"].copy()
print(f"Scientific notation answers: {len(sci)}")
print(f"By domain: {sci['prefix'].value_counts().to_dict()}")
print()

# Classify notation format
def classify_sci_format(ans):
    ans = str(ans)
    if "10^" in ans and "×" in ans:
        return "N × 10^M (caret + ×)"
    if "10^" in ans and "*" in ans:
        return "N * 10^M (caret + *)"
    if "10^" in ans:
        return "N 10^M (caret, other)"
    if re.search(r"10[\u2070-\u2079\u207B]", ans):
        if "×" in ans:
            return "N × 10ˢᵘᵖ (superscript + ×)"
        return "N 10ˢᵘᵖ (superscript, other)"
    if "x 10" in ans.lower():
        return "N x 10^M (lowercase x)"
    return "other"

sci["sci_format"] = sci["answer"].apply(classify_sci_format)
print("=== Scientific Notation Formats ===")
for fmt, cnt in sci["sci_format"].value_counts().items():
    example = sci[sci["sci_format"] == fmt]["answer"].iloc[0]
    print(f"  {fmt:35s} {cnt:4d}  e.g. {example}")

---
## 8. Token Budget Estimation

In [ ]:
# Rough estimation: 1 token ≈ 4 characters for English text
df["q_tokens_est"] = df["q_len"] / 4
df["cot_tokens_est"] = df["cot_len"] / 4
df["total_tokens_est"] = df["q_tokens_est"] + df["cot_tokens_est"]

print("=== Token Budget Estimation (chars/4) ===")
for col, label in [("q_tokens_est", "Question"), ("cot_tokens_est", "CoT"), ("total_tokens_est", "Total")]:
    print(f"  {label:10s}: mean={df[col].mean():.0f}, p50={df[col].median():.0f}, p95={df[col].quantile(0.95):.0f}, max={df[col].max():.0f}")

print(f"\n  Qwen3.5-4B context: 262,144 tokens")
print(f"  P95 input+output: ~{df['total_tokens_est'].quantile(0.95):.0f} tokens")
print(f"  Remaining for few-shot + system prompt: ~{262144 - df['total_tokens_est'].quantile(0.95):.0f} tokens")
print(f"  → Plenty of room for RAG context, few-shot examples, and tool call logs.")

---
## 9. Duplicate & Edge Case Check

In [ ]:
# Duplicate questions
dup_mask = df["question"].duplicated(keep=False)
dups = df[dup_mask].sort_values("question")
print(f"=== Duplicate Questions: {len(dups)} rows ({dups['question'].nunique()} unique) ===")
for q in dups["question"].unique():
    rows = dups[dups["question"] == q][["id", "answer", "unit"]]
    print(f"\n  Q: \"{q[:120]}...\"")
    for _, r in rows.iterrows():
        print(f"    {r['id']}: answer={r['answer']}, unit={r['unit']}")

In [ ]:
# Negative and zero answers
neg = df[df["answer_numeric"] < 0]
zero = df[df["answer_numeric"] == 0]

print(f"=== Negative answers: {len(neg)} ===")
if len(neg) > 0:
    print(neg[["id", "answer", "unit", "prefix"]].to_string())

print(f"\n=== Zero answers: {len(zero)} ===")
if len(zero) > 0:
    print(zero[["id", "answer", "unit", "prefix"]].head(10).to_string())

In [ ]:
# Vietnamese remnants
viet_in_ans = df[df["answer"].str.contains("lần|Hướng|phía|Độ", na=False, regex=True)]
viet_in_unit = df[df["unit"].str.contains("lần|Độ", na=False, regex=True)]

print(f"=== Vietnamese in Answers: {len(viet_in_ans)} ===")
if len(viet_in_ans) > 0:
    print(viet_in_ans[["id", "answer", "unit"]].to_string())

print(f"\n=== Vietnamese in Units: {len(viet_in_unit)} ===")
if len(viet_in_unit) > 0:
    print(viet_in_unit[["id", "answer", "unit"]].to_string())

---
## 10. Summary & Recommendations

### Key Findings

| # | Finding | Impact | Action |
|---|---|---|---|
| 1 | **28% answers are non-numeric** (sci notation, text, Yes/No, multi-value) | Pipeline CANNOT just output float+unit | Build answer type classifier + routing |
| 2 | **Scientific notation has 2+ incompatible formats** (`10^-3` vs `10⁻³`) | Parsing/evaluation will fail | Standardize to single format |
| 3 | **CHLT is 100% Yes/No** (20 samples) | Different strategy needed | Rule-based or formula-check approach |
| 4 | **THCB has 29% multi-value** (`val1; val2` with `unit1; unit2`) | Special output parsing needed | Multi-value parser |
| 5 | **55 unique units** with encoding inconsistencies | Training noise | Normalize all units |
| 6 | **20x domain imbalance** (LD=397 vs CHLT=20) | Model bias | Data augmentation for small domains |
| 7 | **14 null units, 3 Vietnamese remnants** | Incomplete data | Fix manually |
| 8 | **Token budget is comfortable** (P95 ~350 tokens) | No context window issues | Use generous few-shot + tool logs |
| 9 | **CHLT easiest domain by CoT steps but DDT most complex** | Difficulty varies widely | Domain-aware prompting |
| 10 | **CoT quality uncertain** (machine-generated, known errors) | Training on bad CoT = bad model | Re-generate with commercial LLM + SymPy verify |

In [ ]:
# Final summary table
summary = pd.DataFrame({
    "Domain": sorted(df["prefix"].unique()),
    "Count": [len(df[df["prefix"]==p]) for p in sorted(df["prefix"].unique())],
    "% of Total": [f"{len(df[df['prefix']==p])/len(df)*100:.1f}%" for p in sorted(df["prefix"].unique())],
    "Avg CoT Steps": [df[df["prefix"]==p]["cot_steps"].mean().round(1) for p in sorted(df["prefix"].unique())],
    "% Numeric": [f"{len(df[(df['prefix']==p) & (df['answer_type']=='pure_numeric')])/len(df[df['prefix']==p])*100:.0f}%" for p in sorted(df["prefix"].unique())],
    "Top Unit": [df[df["prefix"]==p]["unit"].value_counts().index[0] if df[df["prefix"]==p]["unit"].notna().any() else "NaN" for p in sorted(df["prefix"].unique())],
    "Key Challenge": [
        "RLC circuits, many formulas",       # CH
        "100% Yes/No, only 20 samples",       # CHLT
        "Most complex, mixed answer types",   # DDT
        "Long questions, field calculations",  # DT
        "48% scientific notation, largest",    # LD
        "10% text answers, energy domain",     # NL
        "Capacitor calcs, straightforward",    # TD
        "29% multi-value, error analysis",     # THCB
    ]
})
summary